# Uddevalla buildings

Start with buildings, then inspect their components and related evidence. Statistics describe surrounding areas, not building occupants.

In [1]:
from sweden_ubem import UBEM
import os

db = UBEM(dataset=os.environ.get('UBEM_NOTEBOOK_DATASET', 'uddevalla'))
db

UBEM(dataset='uddevalla')

## Which layers have loaded?

Check this before interpreting missing values.

In [2]:
db.sources()[['layer', 'loading_status', 'publication_status']].drop_duplicates()

,layer,loading_status,publication_status
0,geometry,loaded,source_snapshot
148,register,loaded,source_snapshot
331,addresses,loaded,source_snapshot
410,properties,loaded,source_snapshot
411,heights,loaded,staging_only
662,epc,loaded,source_snapshot
691,income,loaded,source_snapshot
692,regso,loaded,source_snapshot
693,municipality,loaded,source_snapshot
702,population,loaded,source_snapshot


In [3]:
db.completeness().groupby(['layer', 'loading_status', 'evidence_status'], dropna=False).size().rename('buildings')

layer             loading_status  evidence_status
addresses         loaded          ambiguous             44
                                  available          18541
                                  missing            26715
context_linkage   loaded          missing            45300
epc               loaded          ambiguous              4
                                  available           3970
                                  missing            41326
epc_linkage       loaded          missing            45300
geometry          loaded          available          45300
heights           loaded          available          43185
                                  missing             2115
income            loaded          ambiguous              4
                                  available          45296
municipality      loaded          ambiguous              4
                                  available          45296
population        loaded          ambiguous              4
      

## Find buildings

The default shows verified existing buildings in the selected municipality. Other statuses and supporting outside buildings remain inspectable.

In [4]:
buildings = db.buildings()
print(f'{len(buildings):,} existing buildings')
buildings.head()

45,185 existing buildings


,dataset_id,building_id,municipality_code,selection_role,object_status,status_verified,is_existing,house_number,construction_year,main_building,...,linkage_eligible,component_count,determined_height_component_count,entrance_count,resolved_address_count,property_candidate_count,epc_certificate_count,rejected_epc_candidate_count,purposes,geometry
0,uddevalla,000255d4-6183-481a-881e-87041637d702,1485,primary,Gällande,True,True,143,NaN,False,...,True,1,1,0,0,1,0,2,[Komplementbyggnad],"POLYGON ((318154.678 6469745.284, 318156.354 6..."
1,uddevalla,00026799-89b4-4557-bcf2-d863c6c37f44,1485,primary,Gällande,True,True,2,NaN,False,...,True,1,1,7,7,1,0,0,[Bostad],"POLYGON ((317954.768 6455450.053, 317977.251 6..."
2,uddevalla,00027219-82b9-40a7-be89-cfee51b74726,1485,primary,Gällande,True,True,41,NaN,True,...,True,1,1,5,5,1,0,9,[],"POLYGON ((321455.505 6472165.92, 321556.369 64..."
3,uddevalla,0002e256-0211-4148-98e4-644327a96b5e,1485,primary,Gällande,True,True,1,1949.0,False,...,True,1,1,1,1,1,0,0,[Bostad],"POLYGON ((317780.15 6463072.813, 317787.872 64..."
4,uddevalla,0005f223-d3e5-4a98-bd32-c71eb7ca082b,1485,primary,Gällande,True,True,2,NaN,False,...,True,1,1,0,0,2,0,0,[Komplementbyggnad],"POLYGON ((319940.452 6470332.064, 319940.538 6..."


In [5]:
all_records = db.buildings(include_support=True, include_other_statuses=True)
all_records[['selection_role', 'object_status', 'component_count']].head()

,selection_role,object_status,component_count
0,primary,Gällande,1
1,primary,Gällande,1
2,primary,Gällande,1
3,primary,Gällande,1
4,primary,Gällande,1


In [6]:
homes = db.buildings(purpose='Bostad')
homes.head()

,dataset_id,building_id,municipality_code,selection_role,object_status,status_verified,is_existing,house_number,construction_year,main_building,...,linkage_eligible,component_count,determined_height_component_count,entrance_count,resolved_address_count,property_candidate_count,epc_certificate_count,rejected_epc_candidate_count,purposes,geometry
0,uddevalla,00026799-89b4-4557-bcf2-d863c6c37f44,1485,primary,Gällande,True,True,2,NaN,False,...,True,1,1,7,7,1,0,0,[Bostad],"POLYGON ((317954.768 6455450.053, 317977.251 6..."
1,uddevalla,0002e256-0211-4148-98e4-644327a96b5e,1485,primary,Gällande,True,True,1,1949.0,False,...,True,1,1,1,1,1,0,0,[Bostad],"POLYGON ((317780.15 6463072.813, 317787.872 64..."
2,uddevalla,00077d9d-9a04-4900-8f16-31c1de559fbf,1485,primary,Gällande,True,True,1,1955.0,False,...,True,1,1,1,1,1,0,0,[Bostad],"POLYGON ((315752.447 6461737.991, 315754.632 6..."
3,uddevalla,0008b0f9-b1cf-4c79-a5c2-167d333451b2,1485,primary,Gällande,True,True,1,1909.0,False,...,True,1,1,1,1,1,0,0,[Bostad],"POLYGON ((320671.559 6457290.251, 320672.888 6..."
4,uddevalla,001124ef-f7d2-4026-9159-b6471ae8489e,1485,primary,Gällande,True,True,1,1972.0,False,...,True,1,1,1,1,1,0,0,[Bostad],"POLYGON ((314354.017 6471391.83, 314366.326 64..."


## Open one building

Use an LM UUID from the table. This example picks a building with multiple components where available.

In [7]:
if buildings.empty:
    raise RuntimeError('No verified existing buildings: inspect sources and statuses first')
example = buildings.sort_values('component_count', ascending=False).iloc[0]
building = db.building(str(example.building_id))
building

Building(db=UBEM(dataset='uddevalla'), building_id='0e1f0a87-2b90-4eef-824e-307781e6c564')

In [8]:
building.components()

,dataset_id,component_id,building_id,geometry_hash,outline_labels,representation_status,owner_tile_id,source_fids,revision_timestamp,object_version,run_id,geometry
0,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564:00bc554eb...,0e1f0a87-2b90-4eef-824e-307781e6c564,00bc554eb5f1679c32bd4d54ddba3bf43b4038a9d269f3...,[TAKKANT],not_assessed,sweref_3006_x0320000_y6470000,[5749708],2026-01-30 16:36:37.847000+01:00,5,ed425e03-345f-4596-9766-4a25aebc8224,"POLYGON ((320269.608 6472389.768, 320275.585 6..."
1,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564:05d4dfaf0...,0e1f0a87-2b90-4eef-824e-307781e6c564,05d4dfaf06716ef92f3bf6ba9a7207d0457a6fcec810b9...,[FASAD],not_assessed,sweref_3006_x0320000_y6470000,[5749700],2026-01-30 16:36:37.847000+01:00,5,ed425e03-345f-4596-9766-4a25aebc8224,"POLYGON ((320242.006 6472445.04, 320242.631 64..."
2,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564:08dc1a59c...,0e1f0a87-2b90-4eef-824e-307781e6c564,08dc1a59ccd057e4a3d7769df5438f85026cba86f2daf4...,[TAKKANT],not_assessed,sweref_3006_x0320000_y6470000,[5749689],2026-01-30 16:36:37.847000+01:00,5,ed425e03-345f-4596-9766-4a25aebc8224,"POLYGON ((320381.787 6472408.224, 320385.155 6..."
3,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564:13dc6246f...,0e1f0a87-2b90-4eef-824e-307781e6c564,13dc6246ff65759a427a2fd4f6136a8a57993a64b701a6...,[TAKKANT],not_assessed,sweref_3006_x0320000_y6470000,[5749688],2026-01-30 16:36:37.847000+01:00,5,ed425e03-345f-4596-9766-4a25aebc8224,"POLYGON ((320208.547 6472480.519, 320232.9 647..."
4,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564:17248af47...,0e1f0a87-2b90-4eef-824e-307781e6c564,17248af475f83e2d97e043db015027549f45c1874a3134...,[TAKKANT],not_assessed,sweref_3006_x0320000_y6470000,[5749705],2026-01-30 16:36:37.847000+01:00,5,ed425e03-345f-4596-9766-4a25aebc8224,"POLYGON ((320279.584 6472469.83, 320284.909 64..."
5,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564:282b4e133...,0e1f0a87-2b90-4eef-824e-307781e6c564,282b4e1333b8839a5ebc7c14fdd020b29eb02e9955c505...,[TAKKANT],not_assessed,sweref_3006_x0320000_y6470000,[5749684],2026-01-30 16:36:37.847000+01:00,5,ed425e03-345f-4596-9766-4a25aebc8224,"POLYGON ((320321.088 6472406.772, 320370.946 6..."
6,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564:28a89a2df...,0e1f0a87-2b90-4eef-824e-307781e6c564,28a89a2dffa037c713edbf276da7b9867bc2b48025bbae...,[TAKKANT],not_assessed,sweref_3006_x0320000_y6470000,[5749712],2026-01-30 16:36:37.847000+01:00,5,ed425e03-345f-4596-9766-4a25aebc8224,"POLYGON ((320340.044 6472382.358, 320343.615 6..."
7,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564:28ad731c6...,0e1f0a87-2b90-4eef-824e-307781e6c564,28ad731c63dfd6b6c70899dfa904b6f9a235184705aa83...,[TAKKANT],not_assessed,sweref_3006_x0320000_y6470000,[5749694],2026-01-30 16:36:37.847000+01:00,5,ed425e03-345f-4596-9766-4a25aebc8224,"POLYGON ((320364.238 6472420.44, 320371.988 64..."
8,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564:2fb53d6a8...,0e1f0a87-2b90-4eef-824e-307781e6c564,2fb53d6a85fe9d607f3dd693234b008667b9b6d961a883...,[TAKKANT],not_assessed,sweref_3006_x0320000_y6470000,[5749695],2026-01-30 16:36:37.847000+01:00,5,ed425e03-345f-4596-9766-4a25aebc8224,"POLYGON ((320312.651 6472399.556, 320319.455 6..."
9,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564:4364b039e...,0e1f0a87-2b90-4eef-824e-307781e6c564,4364b039eaee7a90a9ae54eb58d40a8baf709a9a906b7d...,[TAKKANT],not_assessed,sweref_3006_x0320000_y6470000,[5749697],2026-01-30 16:36:37.847000+01:00,5,ed425e03-345f-4596-9766-4a25aebc8224,"POLYGON ((320337.266 6472493.393, 320338.147 6..."


In [9]:
building.heights()

,dataset_id,building_id,component_id,input_geometry_hash,height_status,representative_height_m,eave_height_m,ridge_height_m,roof_form,details,evidence_sha256,run_id
0,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,0e1f0a87-2b90-4eef-824e-307781e6c564:00bc554eb...,00bc554eb5f1679c32bd4d54ddba3bf43b4038a9d269f3...,determined,10.155,8.975834,11.568724,pitched_ridge,"{'roof_form': 'pitched_ridge', 'building_id': ...",0671813a15d009e42298e2c0b9f04b1475db493f6ce931...,51631ab5-1db6-40c9-b1db-d8979a6773cf
1,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,0e1f0a87-2b90-4eef-824e-307781e6c564:05d4dfaf0...,05d4dfaf06716ef92f3bf6ba9a7207d0457a6fcec810b9...,not_determined,NaN,NaN,NaN,unclear,"{'roof_form': 'unclear', 'building_id': '0e1f0...",6aae6fdf3f1824f430bfc3451ec33187f93d81e683bc3d...,51631ab5-1db6-40c9-b1db-d8979a6773cf
2,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,0e1f0a87-2b90-4eef-824e-307781e6c564:08dc1a59c...,08dc1a59ccd057e4a3d7769df5438f85026cba86f2daf4...,determined,15.070,14.069339,15.636061,pitched_ridge,"{'roof_form': 'pitched_ridge', 'building_id': ...",6aae6fdf3f1824f430bfc3451ec33187f93d81e683bc3d...,51631ab5-1db6-40c9-b1db-d8979a6773cf
3,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,0e1f0a87-2b90-4eef-824e-307781e6c564:13dc6246f...,13dc6246ff65759a427a2fd4f6136a8a57993a64b701a6...,determined,5.260,NaN,NaN,unclear,"{'roof_form': 'unclear', 'building_id': '0e1f0...",6aae6fdf3f1824f430bfc3451ec33187f93d81e683bc3d...,51631ab5-1db6-40c9-b1db-d8979a6773cf
4,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,0e1f0a87-2b90-4eef-824e-307781e6c564:17248af47...,17248af475f83e2d97e043db015027549f45c1874a3134...,determined,8.070,7.330092,8.989237,pitched_ridge,"{'roof_form': 'pitched_ridge', 'building_id': ...",6aae6fdf3f1824f430bfc3451ec33187f93d81e683bc3d...,51631ab5-1db6-40c9-b1db-d8979a6773cf
5,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,0e1f0a87-2b90-4eef-824e-307781e6c564:282b4e133...,282b4e1333b8839a5ebc7c14fdd020b29eb02e9955c505...,determined,53.880,NaN,NaN,unclear,"{'roof_form': 'unclear', 'building_id': '0e1f0...",6aae6fdf3f1824f430bfc3451ec33187f93d81e683bc3d...,51631ab5-1db6-40c9-b1db-d8979a6773cf
6,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,0e1f0a87-2b90-4eef-824e-307781e6c564:28a89a2df...,28a89a2dffa037c713edbf276da7b9867bc2b48025bbae...,determined,20.575,NaN,NaN,unclear,"{'roof_form': 'unclear', 'building_id': '0e1f0...",6aae6fdf3f1824f430bfc3451ec33187f93d81e683bc3d...,51631ab5-1db6-40c9-b1db-d8979a6773cf
7,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,0e1f0a87-2b90-4eef-824e-307781e6c564:28ad731c6...,28ad731c63dfd6b6c70899dfa904b6f9a235184705aa83...,determined,7.200,NaN,NaN,unclear,"{'roof_form': 'unclear', 'building_id': '0e1f0...",6aae6fdf3f1824f430bfc3451ec33187f93d81e683bc3d...,51631ab5-1db6-40c9-b1db-d8979a6773cf
8,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,0e1f0a87-2b90-4eef-824e-307781e6c564:2fb53d6a8...,2fb53d6a85fe9d607f3dd693234b008667b9b6d961a883...,determined,6.040,NaN,NaN,unclear,"{'roof_form': 'unclear', 'building_id': '0e1f0...",6aae6fdf3f1824f430bfc3451ec33187f93d81e683bc3d...,51631ab5-1db6-40c9-b1db-d8979a6773cf
9,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,0e1f0a87-2b90-4eef-824e-307781e6c564:4364b039e...,4364b039eaee7a90a9ae54eb58d40a8baf709a9a906b7d...,determined,31.120,NaN,NaN,unclear,"{'roof_form': 'unclear', 'building_id': '0e1f0...",6aae6fdf3f1824f430bfc3451ec33187f93d81e683bc3d...,51631ab5-1db6-40c9-b1db-d8979a6773cf


In [10]:
building.addresses()

,dataset_id,building_id,entrance_id,address_place_id,street_name,address_number,locality,resolved,original_values,run_id,geometry
0,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,6df268e2-4476-4578-9b72-0ebd7c633fec,bc1190f3-d9da-411c-aea6-9c1e7097bc6c,Fjällvägen,9,Uddevalla,True,"{'Entre/hiss': ['Uppgift saknas'], 'Entre/entr...",3f7c7392-8207-4c8f-9b2b-6b0e7b1922e8,POINT (320367.83 6472467.873)


In [11]:
building.properties()

,dataset_id,building_id,property_unit_id,intersection_area_m2,overlap_fraction,rank,is_primary,run_id
0,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,909a6a67-e63b-90ec-e040-ed8f66444c3f,12773.743906,1.0,1,True,15e59033-9fae-4ebe-a5c4-205bf04412e0


## Certificates

These relations are accepted matches. Quantities remain certificate-level; rejected candidates are separate.

If geometry representation is unassessed, automatic matches are withheld. An empty building certificate table does not establish that no certificate exists. Check the quality findings and layer coverage below.

In [12]:
building.epc_certificates()

""


In [13]:
building.epc_candidates()

,dataset_id,building_id,subject_id,certificate_id,property_unit_id,decision,evidence_tier,reasons,details,run_id
0,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,267717bfaf07187f0d3b24908b9b58ca1b43558ea5fe83...,efter_v260:1013496,909a6a67-e63b-90ec-e040-ed8f66444c3f,rejected,None,[house_number_disagreement],"{'match_key': '1485|ANNEGRETEBERG 1', 'match_m...",e0e9c43b-8b60-4451-a852-1e79adecf0b7
1,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,886d8f11bf6c3ad842dc1373611ee113fcac9a815a345a...,efter_v260:1024962,909a6a67-e63b-90ec-e040-ed8f66444c3f,rejected,None,[house_number_disagreement],"{'match_key': '1485|ANNEGRETEBERG 1', 'match_m...",e0e9c43b-8b60-4451-a852-1e79adecf0b7
2,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,d13a9f775d87ba84c72004d77e2c7dab7781a12535b330...,efter_v260:1024959,909a6a67-e63b-90ec-e040-ed8f66444c3f,rejected,None,"[house_number_conflict, shared_address_distinc...","{'address_key': ['1485', 'FJÄLLVÄGEN', '9', ''...",e0e9c43b-8b60-4451-a852-1e79adecf0b7


## Statistical context

Empty area tables can mean the layer is not loaded. Do not read grid population as building occupancy. Unresolved building geometry retains positive overlaps but prevents an asserted primary area, so primary-area fields in its context summary can remain unset.

In [14]:
db.boundaries()

,dataset_id,area_type,area_code,name,reference_period,run_id,geometry
0,uddevalla,municipality,1485,Municipality 1485,2025,c9bb7b01-fde0-4348-bfe1-d34cc85ca924,"POLYGON ((315015.59 6457046.427, 314958.306 64..."


In [15]:
db.areas('regso')

,dataset_id,area_type,area_code,name,reference_period,run_id,geometry
0,uddevalla,regso,1485R001,Ammenäs-Forshälla-Backamo,2025,eda12103-f960-44de-8537-d5ee6974738c,"POLYGON ((317526.194 6469462.36, 317569.493 64..."
1,uddevalla,regso,1485R002,Bokenäset-Berg-Sund,2025,eda12103-f960-44de-8537-d5ee6974738c,"POLYGON ((301628.108 6475319.647, 302152.692 6..."
2,uddevalla,regso,1485R003,Hedegärde-Äsperöd-Gräskärr,2025,eda12103-f960-44de-8537-d5ee6974738c,"POLYGON ((322284.871 6474144.017, 322266.293 6..."
3,uddevalla,regso,1485R013,Skansberget-Norra centrum-Sörvik,2025,eda12103-f960-44de-8537-d5ee6974738c,"POLYGON ((319136.492 6472513.828, 319261.936 6..."
4,uddevalla,regso,1485R004,Helenedal-Södra Dalaberg-Bleket,2025,eda12103-f960-44de-8537-d5ee6974738c,"POLYGON ((321787.737 6475649.08, 321822.494 64..."
5,uddevalla,regso,1485R014,Skogslyckan-Underås,2025,eda12103-f960-44de-8537-d5ee6974738c,"POLYGON ((320045.621 6473389.8, 320035.025 647..."
6,uddevalla,regso,1485R005,Herrestad-Sunningen-Utby,2025,eda12103-f960-44de-8537-d5ee6974738c,"POLYGON ((313621.705 6472243.524, 313957.846 6..."
7,uddevalla,regso,1485R006,Hogstorp-Lanesund-Kissleberg,2025,eda12103-f960-44de-8537-d5ee6974738c,"POLYGON ((322634.305 6482103.147, 322586.875 6..."
8,uddevalla,regso,1485R007,Hälle-Lyckorna,2025,eda12103-f960-44de-8537-d5ee6974738c,"POLYGON ((318617.144 6457883.846, 318958.25 64..."
9,uddevalla,regso,1485R008,Kapelle-Walkesborg-Gustafsberg,2025,eda12103-f960-44de-8537-d5ee6974738c,"POLYGON ((319968.325 6470908.025, 319983.695 6..."


In [16]:
db.population_grid()

,dataset_id,area_type,area_code,name,reference_period,run_id,geometry
0,uddevalla,population_grid_1km,2930006463000,None,20251231,a633389c-6237-4b40-b6cc-a2b527ecfbfe,"POLYGON ((293000.002 6462999.995, 293000.002 6..."
1,uddevalla,population_grid_1km,2950006467000,None,20251231,a633389c-6237-4b40-b6cc-a2b527ecfbfe,"POLYGON ((294999.999 6467000.001, 294999.999 6..."
2,uddevalla,population_grid_1km,2950006460000,None,20251231,a633389c-6237-4b40-b6cc-a2b527ecfbfe,"POLYGON ((294999.999 6459999.996, 294999.999 6..."
3,uddevalla,population_grid_1km,2950006461000,None,20251231,a633389c-6237-4b40-b6cc-a2b527ecfbfe,"POLYGON ((294999.999 6461000.002, 294999.999 6..."
4,uddevalla,population_grid_1km,2940006462000,None,20251231,a633389c-6237-4b40-b6cc-a2b527ecfbfe,"POLYGON ((294000.001 6461999.999, 294000.001 6..."
...,...,...,...,...,...,...,...
582,uddevalla,population_grid_1km,3310006474000,None,20251231,a633389c-6237-4b40-b6cc-a2b527ecfbfe,"POLYGON ((331000.003 6473999.996, 331000.003 6..."
583,uddevalla,population_grid_1km,3310006475000,None,20251231,a633389c-6237-4b40-b6cc-a2b527ecfbfe,"POLYGON ((331000.003 6475000.002, 331000.003 6..."
584,uddevalla,population_grid_1km,3310006476000,None,20251231,a633389c-6237-4b40-b6cc-a2b527ecfbfe,"POLYGON ((331000.003 6475999.999, 331000.003 6..."
585,uddevalla,population_grid_1km,3310006477000,None,20251231,a633389c-6237-4b40-b6cc-a2b527ecfbfe,"POLYGON ((331000.003 6476999.995, 331000.003 6..."


In [17]:
building.context_summary()

,dataset_id,building_id,municipality,regso,population_cell,primary_observations
0,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,1485,1485R014,3200006472000,"[{'unit': 'person', 'value': 3075, 'period': '..."


In [18]:
building.context()

,dataset_id,building_id,area_type,area_code,is_primary,assignment_status,observation_id,indicator_id,reference_period,dimensions,raw_value,normalized_value,source_unit,normalized_unit,transformation,value_status,provenance,run_id
0,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,population_grid_1km,3200006472000,True,resolved,b657f6e35e7da3999d0dea4af743fd346678aad1ca2664...,scb:population:beftotalt,20251231,{'grid_size_m': '1000'},3075,3075.0,person,person,identity,published_protected,"{'fid': '4623', 'file': '/Users/ssanjay/GitHub...",a633389c-6237-4b40-b6cc-a2b527ecfbfe
1,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,population_grid_1km,3200006472000,True,resolved,b0c5e716b219c28db15555ad9588c65f42d966ba10ddbf...,scb:population:man,20251231,{'grid_size_m': '1000'},1587,1587.0,person,person,identity,published_protected,"{'fid': '4623', 'file': '/Users/ssanjay/GitHub...",a633389c-6237-4b40-b6cc-a2b527ecfbfe
2,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,population_grid_1km,3200006472000,True,resolved,1e8f77627a0c97b9eaa37b05be2a85caa684f5f27b550f...,scb:population:kvinna,20251231,{'grid_size_m': '1000'},1490,1490.0,person,person,identity,published_protected,"{'fid': '4623', 'file': '/Users/ssanjay/GitHub...",a633389c-6237-4b40-b6cc-a2b527ecfbfe
3,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,population_grid_1km,3200006472000,True,resolved,81d08886cf3322d60becb8c64a019657e84419f884695c...,scb:population:ald0_4,20251231,{'grid_size_m': '1000'},107,107.0,person,person,identity,published_protected,"{'fid': '4623', 'file': '/Users/ssanjay/GitHub...",a633389c-6237-4b40-b6cc-a2b527ecfbfe
4,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,population_grid_1km,3200006472000,True,resolved,b65fb6f9cfbed1fafbabfbaf3cf09ee7df435e6512d091...,scb:population:ald5_9,20251231,{'grid_size_m': '1000'},111,111.0,person,person,identity,published_protected,"{'fid': '4623', 'file': '/Users/ssanjay/GitHub...",a633389c-6237-4b40-b6cc-a2b527ecfbfe
5,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,population_grid_1km,3200006472000,True,resolved,13074c24bce7e5d9e608ea0db89ed39f0a5f98e7ec00cd...,scb:population:ald10_14,20251231,{'grid_size_m': '1000'},126,126.0,person,person,identity,published_protected,"{'fid': '4623', 'file': '/Users/ssanjay/GitHub...",a633389c-6237-4b40-b6cc-a2b527ecfbfe
6,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,population_grid_1km,3200006472000,True,resolved,24b332c1a9e65aa17f46b964a647abff1e8a48159f4bca...,scb:population:ald15_19,20251231,{'grid_size_m': '1000'},162,162.0,person,person,identity,published_protected,"{'fid': '4623', 'file': '/Users/ssanjay/GitHub...",a633389c-6237-4b40-b6cc-a2b527ecfbfe
7,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,population_grid_1km,3200006472000,True,resolved,bedc6de10b53e4399c5c751f18429eb1d017e8a75eb84e...,scb:population:ald20_24,20251231,{'grid_size_m': '1000'},227,227.0,person,person,identity,published_protected,"{'fid': '4623', 'file': '/Users/ssanjay/GitHub...",a633389c-6237-4b40-b6cc-a2b527ecfbfe
8,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,population_grid_1km,3200006472000,True,resolved,a3cd4483397c6f31d5839c276337e94478d56d7eff2b40...,scb:population:ald25_29,20251231,{'grid_size_m': '1000'},287,287.0,person,person,identity,published_protected,"{'fid': '4623', 'file': '/Users/ssanjay/GitHub...",a633389c-6237-4b40-b6cc-a2b527ecfbfe
9,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,population_grid_1km,3200006472000,True,resolved,68aa4bd554edd1771de96239f13ac4e3592c7d0c0cb20b...,scb:population:ald30_34,20251231,{'grid_size_m': '1000'},271,271.0,person,person,identity,published_protected,"{'fid': '4623', 'file': '/Users/ssanjay/GitHub...",a633389c-6237-4b40-b6cc-a2b527ecfbfe


In [19]:
building.completeness()

,dataset_id,building_id,layer,loading_status,evidence_status,available_count,required_count,reasons
0,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,geometry,loaded,available,32,1,[]
1,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,register,loaded,available,1,1,[]
2,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,addresses,loaded,available,1,1,[]
3,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,properties,loaded,available,1,1,[]
4,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,heights,loaded,available,31,32,[]
5,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,epc,loaded,missing,0,1,[]
6,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,municipality,loaded,available,1,1,[]
7,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,population,loaded,available,1,1,[]
8,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,regso,loaded,available,1,1,[]
9,uddevalla,0e1f0a87-2b90-4eef-824e-307781e6c564,income,loaded,available,1,1,[]


In [20]:
db.quality().head()

,finding_id,dataset_id,building_id,layer,code,details,run_id
0,1,uddevalla,19b2d684-db17-4a42-b34c-7057d87ea925,geometry,lm_municipality_conflict,"{'selected_municipality': '1485', 'source_muni...",ed425e03-345f-4596-9766-4a25aebc8224
1,2,uddevalla,20b7d5d6-3e06-4162-a80b-e2aa37ba05c9,geometry,lm_municipality_conflict,"{'selected_municipality': '1485', 'source_muni...",ed425e03-345f-4596-9766-4a25aebc8224
2,3,uddevalla,c124506e-357e-4f1d-b95d-d4a588b7af6e,geometry,lm_municipality_conflict,"{'selected_municipality': '1485', 'source_muni...",ed425e03-345f-4596-9766-4a25aebc8224
3,4,uddevalla,e38b5dcc-8ce1-41de-b26e-54c2322d711c,geometry,lm_municipality_conflict,"{'selected_municipality': '1485', 'source_muni...",ed425e03-345f-4596-9766-4a25aebc8224
4,5,uddevalla,dbd31e5a-a628-4b7f-b1a4-90542fe58daf,register,record_missing,{},3f7c7392-8207-4c8f-9b2b-6b0e7b1922e8


## Export a selection

Export writes shared certificates once and includes relationship tables. Choose a new output directory before running:

```python
db.export(homes, format='geopackage', output='output/exports/homes')
```

This cell does not write an export automatically.